In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()
url = os.environ["DATABASE_URL"].replace("postgresql://", "postgresql+psycopg://")
engine = create_engine(url, pool_pre_ping=True)

## Raw neighborhood values — how messy is the location field before normalization?

In [2]:
pd.read_sql("""
    SELECT raw_neighborhood, COUNT(*) AS n
    FROM listings GROUP BY raw_neighborhood ORDER BY n DESC
""", engine)

,raw_neighborhood,n
0,"Caballito, Capital Federal",165
1,"Palermo, Capital Federal",160
2,"Belgrano, Capital Federal",133
3,"Villa Urquiza, Capital Federal",120
4,"Recoleta, Capital Federal",118
...,...,...
71,"Parque Avellaneda, Capital Federal",1
72,"Parque Las Heras, Barrio Norte",1
73,"Boca, Capital Federal",1
74,"River, Nuñez",1


76 distinct raw values, messy two ways: trailing ', Capital Federal' and sub-zones tagged to a parent (e.g. 'Palermo Hollywood, Palermo'). This is what normalization cleans up.


## Currency mix — what currency are prices in?

In [10]:
pd.read_sql("""
    SELECT currency, COUNT(*) AS n
    FROM listing_snapshots GROUP BY currency ORDER BY n DESC
""", engine)

,currency,n
0,USD,2292
1,NaN,8


Prices are USD (2292 snapshots), 8 blank. Sales are quoted in USD, so the analysis filters to currency = 'USD'.

## Area completeness — which m² column is usable?


In [4]:
pd.read_sql("""
    SELECT COUNT(*) AS total,
           COUNT(covered_m2) AS has_covered,
           COUNT(total_m2) AS has_total,
           COUNT(*) FILTER (WHERE covered_m2 > 0) AS covered_positive
    FROM listings
""", engine)

,total,has_covered,has_total,covered_positive
0,1994,1907,0,1907


total_m2 is empty (0 rows), so covered_m2 is the denominator

## Pozo feasibility — is construction status captured?

In [5]:
pd.read_sql("SELECT is_pozo, COUNT(*) AS n FROM listings GROUP BY is_pozo", engine)

,is_pozo,n
0,None,1994


## Pozo recovery — any text to infer it from?

In [6]:
pd.read_sql("SELECT COUNT(*) AS total, COUNT(description) AS has_description FROM listings", engine)

,total,has_description
0,1994,0


is_pozo and description are both empty, so pozo-vs-finished isn't possible with this data

## Room-count completeness

In [7]:
pd.read_sql("""
    SELECT COUNT(*) AS total,
           COUNT(ambientes) AS has_ambientes,
           COUNT(bedrooms) AS has_bedrooms,
           COUNT(bathrooms) AS has_bathrooms
    FROM listings
""", engine)

,total,has_ambientes,has_bedrooms,has_bathrooms
0,1994,1977,1712,409


Ambientes 99% populated, bathrooms only 21%, so bathrooms is unusable

## Post-normalization check — every listing got a clean barrio?

In [8]:
pd.read_sql("""
    SELECT neighborhood, COUNT(*) AS n
    FROM listings GROUP BY neighborhood ORDER BY n DESC
""", engine)

,neighborhood,n
0,Palermo,263
1,Caballito,211
2,Belgrano,189
3,Recoleta,183
4,Balvanera,144
5,Villa Urquiza,124
6,Almagro,106
7,Villa Devoto,84
8,Villa Crespo,66
9,Flores,64


The 76 raw values collapse to 41 clean barrios; counts sum to 1994 (every listing), no NULLs. Normalization mapped everything with no losses.

## The ranking — price per m², total price, size and rooms by barrio:

In [9]:
pd.read_sql("""
    WITH latest AS (
        SELECT DISTINCT ON (listing_id) listing_id, price, currency
        FROM listing_snapshots
        ORDER BY listing_id, captured_at DESC
    )
    SELECT l.neighborhood,
           COUNT(*) AS n,
           ROUND(percentile_cont(0.5) WITHIN GROUP (ORDER BY s.price / l.covered_m2)) AS median_usd_m2,
           ROUND(percentile_cont(0.5) WITHIN GROUP (ORDER BY s.price))                AS median_price,
           ROUND(percentile_cont(0.5) WITHIN GROUP (ORDER BY l.covered_m2))           AS median_m2,
           mode() WITHIN GROUP (ORDER BY l.ambientes)                                 AS mode_amb
    FROM listings l
    JOIN latest s ON s.listing_id = l.id
    WHERE s.currency = 'USD' AND l.covered_m2 > 0
      AND s.price IS NOT NULL AND l.neighborhood IS NOT NULL
    GROUP BY l.neighborhood
    HAVING COUNT(*) >= 10
    ORDER BY median_usd_m2 DESC
""", engine)

,neighborhood,n,median_usd_m2,median_price,median_m2,mode_amb
0,Puerto Madero,12,5788.0,612500.0,120.0,3
1,Villa Devoto,79,3157.0,205000.0,71.0,3
2,Palermo,246,3129.0,165000.0,52.0,2
3,Saavedra,30,3069.0,127801.0,42.0,2
4,Núñez,49,3014.0,165000.0,56.0,3
5,Villa Urquiza,122,2921.0,135000.0,49.0,3
6,Belgrano,187,2889.0,185000.0,62.0,2
7,Colegiales,45,2875.0,167000.0,60.0,3
8,Villa Pueyrredón,17,2725.0,142000.0,51.0,2
9,Chacarita,11,2706.0,90000.0,33.0,1


Median price per m², median total price, median size, and modal ambientes by barrio. USD only, n ≥ 10, ranked by price per m².
